## Exploration

In [1]:
import re
import sys

import pandas as pd

sys.path.insert(0, "..")  # so `import verano` works from notebooks/
from verano.load import load_raw

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

scraped, calls, crm = load_raw()
print(f"scrape {scraped.shape}   calls {calls.shape}   crm {crm.shape}")

scrape (3498, 22)   calls (1880, 12)   crm (508, 13)


In [2]:
scraped.head()

,source_row_id,source_platform,scraped_date,name,address,city,postcode,neighbourhood,phone,website,...,rating,review_count,fullservice_keyword_filter,fullservice_llm_filter,fullservice_llm_confidence,pos_system,pos_signal,pos_eligibility,chain_size_bucket,restaurant_group
0,R002103,bookwell,2026-07-27,Sixpence House,1658 W Prospect Ave,Verano Bay,40740,Greenhithe,(555) 437-4643,https://www.sixpencehouse.example,...,4.4,1340,pass,pass,0.88,NaN,none,unknown,SME,NaN
1,R002581,tablefind,2026-07-20,Juniper Food Truck,1643 W Vance Rd,Verano Bay,40756,Alder Heights,+1 555 642 2905,https://www.juniperfoodtruck.example,...,3.6,532,pass,fail,0.76,not found,none,unknown,SME,NaN
2,R001050,dineline,2026-07-12,Golden Hour Bistro,3160 S Ashfield Ave,Verano Bay,40783,Stillwater,(555) 704-6305,NaN,...,3.5,1681,pass,pass,0.82,NaN,none,unknown,SME,NaN
3,R002315,tablefind,2026-07-23,Hearth Bakery,2271 E Tarrow Road,Verano Bay,40276,Rivera District,+15555499791,NaN,...,4.1,1947,fail,fail,0.9,not found,none,unknown,SME,NaN
4,R003191,tablefind,2026-07-26,Wildwood Brasserie,7293 W Prospect St,Verano Bay,40139,Templeton,+15559849881,https://www.wildwoodbrasserie.example,...,4.4,118,pass,pass,0.89,Northstar,website_fingerprint,pass,mid-cap,Wildwood Group


In [3]:
scraped['pos_system'].value_counts()

pos_system
not found               370
Unknown                 348
-                       347
UNKNOWN                 341
unknown                 339
Ironclad POS             43
ironclad                 42
Ironclad                 40
ironcladpos              38
IRONCLAD                 31
cadence                  23
CADENCE                  23
Cadence                  22
Cadence Station          22
Cadence Mini             17
meridian cloud           17
cadence flex             16
Cadence POS              16
vellum pos               16
Meridian Cloud POS       15
vellum                   14
Meridian Cloud           14
Vellum Restaurant        12
Northstar Restaurant     11
saltbox                  11
FERNPOST                 10
Saltbox                  10
VELLUM                   10
rivet                    10
Salt Box                  9
Vellum POS                9
Fernpost                  9
Northstar                 9
Northstar POS             8
Rivet                     8
Meridian 

In [4]:
calls.head()

,call_id,call_date,caller_type,agent_name,restaurant_name,phone,outcome,pos_captured,poc_name,poc_role,availability_window,poc_email
0,C001433,2026-06-11,ai,dialler-02,Nine Bridges Wine Bar,(555) 104-4446,POS Validated,vellum,NaN,NaN,NaN,NaN
1,C001157,2026-06-24,ai,dialler-01,Ironwood Taqueria,5558404844,Not reached,NaN,NaN,NaN,NaN,NaN
2,C001810,2026-07-25,ai,dialler-02,FALLOWFIELD KITCHEN,+1 555 874 1796,POS Validated,IRONCLAD,NaN,NaN,NaN,NaN
3,C000966,2026-06-21,human,Fatima,ROAN TAQUERIA,(555) 512-6514,Data capture incomplete,NaN,NaN,NaN,NaN,NaN
4,C001097,2026-07-13,human,Sana,Thistle Brasserie,+15553485831,No Answer,NaN,NaN,NaN,NaN,NaN


In [5]:
crm.head()

,deal_id,deal_name,deal_stage,deal_owner,amount,create_date,last_activity_date,associated_contact,company_phone,deal_description,pos_from_crm,source,city
0,D100338,Palisade Cantina,KDM Demo,Marco Ferrari,NaN,2026-04-06,2026-05-22,NaN,555-366-2915,1 location,NaN,human_call,Verano Bay
1,D100301,Sparrow Chophouse,Eligible,Marco Ferrari,NaN,2026-07-13,2026-07-24,NaN,(555) 287-6202,"8 locations, called twice, gatekeeper",Cadence Mini,human_call,Verano Bay
2,D100490,Blue Heron Smokehouse,Eligible,Tom Brennan,NaN,2026-05-07,2026-05-07,NaN,(555) 607-2815,"called twice, gatekeeper",NaN,ai_call,Verano Bay
3,D100012,The Heron Chophouse,Eligible,Tom Brennan,NaN,2026-04-22,2026-04-28,NaN,555-383-7725,NaN,NaN,walk-in,Verano Bay
4,D100106,Tin Roof Table,Eligible,Daniel Reyes,NaN,2026-02-12,2026-02-15,NaN,555-270-0214,"ironcladpos, 11 locations",IRONCLAD,referral,Verano Bay


## 1. What is missing

Before anything else: which columns can I actually rely on?

In [6]:
def missing_report(df, name):
    empty = df.isna().sum()
    return pd.DataFrame(
        {"file": name, "missing": empty, "pct": (100 * empty / len(df)).round(1)}
    ).query("missing > 0")


report = pd.concat(
    [
        missing_report(scraped, "scrape"),
        missing_report(calls, "calls"),
        missing_report(crm, "crm"),
    ]
)
print(report)

# This understates the POS gap: alongside 1,118 blanks, the column also holds
# "unknown", "not found" and "-", which mean the same thing.
blank_or_placeholder = scraped["pos_system"].isna() | scraped["pos_system"].isin(
    ["unknown", "Unknown", "UNKNOWN", "not found", "-", "n/a"]
)
print(f"\nscrape rows with no real POS value: {blank_or_placeholder.sum()} of {len(scraped)}")


                       file  missing   pct
neighbourhood        scrape      115   3.3
phone                scrape      151   4.3
website              scrape      733  21.0
pos_system           scrape     1118  32.0
restaurant_group     scrape     3122  89.3
pos_captured          calls     1502  79.9
poc_name              calls     1603  85.3
poc_role              calls     1692  90.0
availability_window   calls     1792  95.3
poc_email             calls     1838  97.8
amount                  crm      507  99.8
associated_contact      crm      355  69.9
deal_description        crm       51  10.0
pos_from_crm            crm      370  72.8

scrape rows with no real POS value: 2863 of 3498


## 2. One row per listing, not per restaurant

Let me look at one restaurant as the six platforms see it.

In [7]:
full_service = scraped[
    (scraped["city"] == "Verano Bay") & (scraped["fullservice_llm_filter"] == "pass")
]
most_listed = full_service["phone"].value_counts().index[0]

full_service[full_service["phone"] == most_listed][
    ["source_platform", "name", "neighbourhood", "rating", "review_count", "pos_system"]
]

,source_platform,name,neighbourhood,rating,review_count,pos_system
465,civicpages,Tin Roof Steakhouse - Ashgrove,Templeton,4.3,645,not found
2196,atlasmaps,Tin Roof Steakhouse,Templeton,4.3,762,-
3012,roamly,TIN ROOF STEAKHOUSE,Templeton,4.3,763,UNKNOWN


Same venue, different names and ratings per platform. Phone is the stable key, normalize it next.

In [8]:
print(scraped["phone"].dropna().sample(8))

# Sketch. The tested version of this goes into verano/clean.py next.
def digits_only(value):
    d = re.sub(r"\D", "", str(value))
    return d[1:] if len(d) == 11 and d.startswith("1") else d


lengths = scraped["phone"].map(digits_only).str.len().value_counts(dropna=False)

print("\ndigit counts after phone number normalization:")
print(lengths.to_string())

772          5553340549
3212         5558815371
1626     (555) 977-2174
1450       555.627.1425
977     +1 555 543 2878
1027       555.323.4308
2681         5556363079
2684     (555) 502-7699
Name: phone, dtype: object

digit counts after phone number normalization:
phone
10    3347
0      151


## 3. Does phone actually join the three systems?

Five formats, one underlying number. If phone is clean enough, it solves the join that the case says was never designed to work.

In [9]:
scrape_phones = set(scraped["phone"].map(digits_only)) - {""}

for label, df, column in [("calls", calls, "phone"), ("crm", crm, "company_phone")]:
    hits = df[column].map(digits_only).isin(scrape_phones).sum()
    print(f"{label}: {hits} of {len(df)} rows match a scrape phone ({100 * hits / len(df):.1f}%)")

print(f"\ndistinct phones in scrape: {len(scrape_phones)}")
print(f"scrape rows with no usable phone: {(scraped['phone'].map(digits_only) == '').sum()}")

calls: 1880 of 1880 rows match a scrape phone (100.0%)
crm: 508 of 508 rows match a scrape phone (100.0%)

distinct phones in scrape: 1917
scrape rows with no usable phone: 151


Phone is the identity, and all three systems reconcile on it.

The gap is the 151 scrape rows with no phone. Those need a fallback.

For the 151 rows with no phone, name is the fallback — but stripping suffixes merges chain branches. Keep `neighbourhood` attached.

In [10]:
def name_sketch(value):
    text = str(value).lower().split(" - ")[0]
    text = text.replace("verano bay", "")
    text = re.sub(r"\brestaurant\b", "", text)
    text = re.sub(r"^the\b", "", text)
    text = re.sub(r"\s+\d+$", "", text.strip())
    return re.sub(r"[^a-z0-9]", "", text)

probe = scraped.assign(nk=scraped["name"].map(name_sketch), pk=scraped["phone"].map(digits_only))
probe = probe[probe["pk"] != ""]

phones_per_name = probe.groupby("nk")["pk"].nunique()
print(f"name keys that map to more than one phone: {(phones_per_name > 1).sum()}")

collision = phones_per_name.idxmax()
probe[probe["nk"] == collision][["name", "neighbourhood", "phone"]].drop_duplicates("phone")

name keys that map to more than one phone: 64


,name,neighbourhood,phone
1859,The Amber Cafe 1,Old Mill,555.714.6649
2209,Amber Cafe,Kingsway,555-653-7967
2978,Amber Cafe 1,Old Mill,5557146649


## 4. Which full-service filter do I trust?


In [11]:
print(pd.crosstab(scraped["fullservice_keyword_filter"], scraped["fullservice_llm_filter"]))

fullservice_llm_filter      fail  pass
fullservice_keyword_filter            
fail                         390     0
pass                         420  2688


In [12]:
confidence = scraped["fullservice_llm_confidence"].astype(float)
print(confidence.describe().round(2).to_string())

# A "fail" the model was unsure about is a different risk from a confident fail.
shaky = scraped[(scraped["fullservice_llm_filter"] == "fail") & (confidence < 0.80)]
print(f"\nlow-confidence rejections (<0.80): {len(shaky)}")

count    3498.00
mean        0.85
std         0.08
min         0.72
25%         0.79
50%         0.86
75%         0.92
max         0.99

low-confidence rejections (<0.80): 229


Every row the keyword filter rejects, the LLM also rejects. On top of that, the LLM rejects 420 rows the keyword filter would have kept.

So, we'll use the LLM as the full-service gate. It is the precision step; the keyword filter adds nothing on its own.

## 5. Our market

The scrape was run area by area, so some rows may have spilled over the boundary.

In [13]:
print(scraped["city"].value_counts().to_string())
print(f"rows with no neighborhood: {scraped['neighbourhood'].isna().sum()}")
print(f"rows with no city: {scraped["city"].isna().sum()}")

city
Verano Bay       3383
Northmoor          28
Calla Ridge        26
Kestrel Falls      23
Port Adair         22
Sundermere         16
rows with no neighborhood: 115
rows with no city: 0


## 6. POS cache is stale

The scrape's `pos_eligibility` column predates Cadence becoming eligible (2026-06-22). Recompute from `config/rules.yml`.

In [14]:
# Cached verdict in the file vs the verdict today's rules produce.

from verano.clean import pos_key
from verano.load import load_pos_rules

rules = load_pos_rules()

fresh = scraped["pos_system"].map(pos_key).map(lambda k: rules.get(k, ("—", "unknown"))[1])

wrong = scraped[(fresh == "eligible") & (scraped["pos_eligibility"] == "fail")]
print(f"\nrows the cache calls ineligible that today's rules call eligible: {len(wrong)}")
print(wrong["pos_system"].value_counts().to_string())


rows the cache calls ineligible that today's rules call eligible: 139
pos_system
CADENCE            23
cadence            23
Cadence Station    22
Cadence            22
Cadence Mini       17
cadence flex       16
Cadence POS        16


**139 Cadence rows** are marked ineligible in the cache but eligible under today's rules. Ignore the cached column.

## 7. Dialer output

Three human agents plus a dialler, two months, roughly 40 hours a week of capacity.

In [15]:
print(pd.crosstab(calls["outcome"], calls["caller_type"], margins=True))
print(f"\ncall dates: {calls['call_date'].min()} to {calls['call_date'].max()}")

caller_type                ai  human   All
outcome                                   
Callback                  171    172   343
Data capture incomplete   155    104   259
Gatekeeper                 76     49   125
No Answer                 258    220   478
Not reached               178    119   297
POS Validated             194    184   378
All                      1032    848  1880

call dates: 2026-06-09 to 2026-08-02


## 8. Six months of CRM, and what came out of it

The CRM has been running since January with five reps working it.

In [16]:
TODAY = pd.Timestamp("2026-08-03")  # the Monday the case is set on

deals = crm.assign(phone=crm["company_phone"].map(digits_only))
deals["idle_days"] = (TODAY - pd.to_datetime(deals["last_activity_date"])).dt.days

stages = deals["deal_stage"].value_counts()
print(stages.to_string())
print(f"\nshare sitting in the entry stage: {100 * stages['Eligible'] / len(deals):.0f}%")
print(f"median days since any activity:   {deals['idle_days'].median():.0f}")
print(f"deals untouched for 30+ days:     {(deals['idle_days'] > 30).sum()} of {len(deals)}")

deal_stage
Eligible           346
Walked-in           63
KDM Demo            34
Gatekeeper Demo     22
Not Qualified       15
Closed Won          13
Closed Lost         10
Verbal Yes           5

share sitting in the entry stage: 68%
median days since any activity:   80
deals untouched for 30+ days:     415 of 508


In [17]:
# Are reps working restaurants we could never sell to?
from verano.clean import pos_key
from verano.load import load_pos_rules

rules = load_pos_rules()

def pos_verdict(value):
    if pd.isna(value):
        return "unknown"
    return rules.get(pos_key(value), ("", "unknown"))[1]

deals["pos_verdict"] = deals["pos_from_crm"].map(pos_verdict)
print(pd.crosstab(deals["deal_stage"], deals["pos_verdict"], margins=True))
print(f"\n{100 * (deals['pos_verdict'] == 'unknown').mean():.0f}% of deals have no POS in pos_from_crm")
print(f"{100 * (deals['pos_verdict'] == 'ineligible').mean():.1f}% of all deals are on ineligible POS")

pos_verdict      eligible  ineligible  unknown  All
deal_stage                                         
Closed Lost             2           1        7   10
Closed Won              1           2       10   13
Eligible               49          43      254  346
Gatekeeper Demo         4           2       16   22
KDM Demo                9           1       24   34
Not Qualified           2           0       13   15
Verbal Yes              0           0        5    5
Walked-in               9          13       41   63
All                    76          62      370  508

73% of deals have no POS in pos_from_crm
12.2% of all deals are on ineligible POS


In [18]:
# Two deals on one restaurant is a collision. Two owners on one restaurant is worse.
repeats = deals[deals.duplicated("phone", keep=False)]
owners_per_restaurant = repeats.groupby("phone")["deal_owner"].nunique()

print(f"restaurants with more than one deal: {repeats['phone'].nunique()}")
print(f"  of those, worked by two different reps: {(owners_per_restaurant > 1).sum()}")

example = owners_per_restaurant[owners_per_restaurant > 1].index[0]
deals[deals["phone"] == example][["deal_name", "deal_stage", "deal_owner", "create_date", "source"]]

restaurants with more than one deal: 38
  of those, worked by two different reps: 25


,deal_name,deal_stage,deal_owner,create_date,source
76,Ironwood Grill,Eligible,Daniel Reyes,2026-05-12,human_call
291,The Ironwood Grill,Eligible,Aisha Karim,2026-06-01,human_call


In [19]:
print(f"deals with pos_from_crm filled: {crm['pos_from_crm'].notna().sum()} of {len(crm)}")
notes_have_pos = crm["deal_description"].fillna("").str.lower().apply(
    lambda note: any(k in pos_key(note) for k in rules)
)
print(f"deals with POS only in free-text notes: {(notes_have_pos & crm['pos_from_crm'].isna()).sum()}")

deals with pos_from_crm filled: 138 of 508
deals with POS only in free-text notes: 126


## Decisions for the pipeline

- **Identity:** normalised 10-digit phone, it joins 100% of call and CRM rows
- **No phone:** name + `neighbourhood` fallback; merge on the one that has phone number
- **Full-service:** LLM filter only, since  keyword filter adds nothing at the precision step
- **Market:** `city == "Verano Bay"`
- **POS rules:** use `config/rules.yml`, ignore cached `pos_eligibility`, where 139 Cadence rows are wrong
- **POS sources:** The pipeline should recomputes every verdict from `config/rules.yml`
- **Unknown POS:** not disqualified data goes to researchers, not reps

**Others**

- **Ratings / reviews / chain size:** aggregate across platforms; use for ranking